# ☕ Starbucks Customer Survey — Visual Analysis

**Author:** Portfolio project · **Stack:** pandas · seaborn · matplotlib · scikit-learn · XGBoost

This notebook explores a Malaysian Starbucks customer satisfaction survey
(*n = 122 respondents, 21 questions*) and answers three business questions:

1. **Who** are Starbucks' customers, and how do they behave?
2. **What** drives their satisfaction across quality, price, ambiance, WiFi and service?
3. **Will they keep buying?** — and can we predict it from the survey?

All visualisations follow a consistent Starbucks-inspired, colour-blind-aware
palette and publication-quality styling.


## 1 · Setup & styling

In [ ]:
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl, seaborn as sns
warnings.filterwarnings("ignore")

# ---------- Brand palette (colour-blind aware) ----------
SB_GREEN, SB_DARK_GREEN, SB_GOLD = "#00704A", "#1E3932", "#CBA258"
SB_BROWN, SB_WARM, SB_CREAM      = "#5A3A22", "#D4A56A", "#F1E9D2"
SB_RED, SB_GRAY                  = "#C0392B", "#7F8C8D"
PALETTE  = [SB_GREEN, SB_GOLD, SB_BROWN, SB_DARK_GREEN, SB_WARM, SB_RED, SB_GRAY, SB_CREAM]
SEQ_CMAP = sns.light_palette(SB_GREEN, as_cmap=True)
DIV_CMAP = sns.diverging_palette(20, 150, s=80, l=45, as_cmap=True)

sns.set_theme(style="whitegrid", context="talk")
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.titleweight": "bold", "axes.titlesize": 14,
    "axes.titlecolor": SB_DARK_GREEN, "axes.spines.top": False,
    "axes.spines.right": False, "axes.edgecolor": "#CCCCCC",
    "grid.color": "#EEEEEE", "legend.frameon": False,
})

def annotate_bars(ax, fmt="{:.0f}", offset=3):
    for p in ax.patches:
        h = p.get_height()
        if pd.isna(h) or h == 0: continue
        ax.annotate(fmt.format(h),
                    (p.get_x()+p.get_width()/2, h),
                    ha="center", va="bottom",
                    xytext=(0, offset), textcoords="offset points",
                    fontsize=9, color="#333")


## 2 · Load & clean the survey

In [ ]:
df = pd.read_csv("Starbucks satisfactory survey.csv")

rename_map = {
    'Timestamp':'timestamp', '1. Your Gender':'gender', '2. Your Age':'age',
    '3. Are you currently....?':'current_status',
    '4. What is your annual income?':'annual_income',
    '5. How often do you visit Starbucks?':'visit_frequency',
    '6. How do you usually enjoy Starbucks?':'visit_type',
    '7. How much time do you normally  spend during your visit?':'visit_duration',
    "8. The nearest Starbucks's outlet to you is...?":'nearest_outlet',
    '9. Do you have Starbucks membership card?':'membership_card',
    '10. What do you most frequently purchase at Starbucks?':'frequent_purchase',
    '11. On average, how much would you spend at Starbucks per visit?':'avg_spend_per_visit',
    '12. How would you rate the quality of Starbucks compared to other brands (Coffee Bean, Old Town White Coffee..) to be:':'quality_rating',
    '13. How would you rate the price range at Starbucks?':'price_rating',
    '14. How important are sales and promotions in your purchase decision?':'promo_importance',
    '15. How would you rate the ambiance at Starbucks? (lighting, music, etc...)':'ambiance_rating',
    '16. You rate the WiFi quality at Starbucks as..':'wifi_quality',
    '17. How would you rate the service at Starbucks? (Promptness, friendliness, etc..)':'service_rating',
    '18. How likely you will choose Starbucks for doing business meetings or hangout with friends?':'meeting_or_hangout_likelihood',
    '19. How do you come to hear of promotions at Starbucks? Check all that apply.':'promo_channels',
    '20. Will you continue buying at Starbucks?':'continue_buying',
}
df = df.rename(columns=rename_map).dropna().drop_duplicates()

# Friendly age labels (ordered)
age_order = ['Teens', 'Young Adults', 'Adults', 'Mature Adults']
df['age_group'] = pd.Categorical(df['age'].map({
    'Below 20':'Teens', 'From 20 to 29':'Young Adults',
    'From 30 to 39':'Adults', '40 and above':'Mature Adults'}),
    categories=age_order, ordered=True)

# Spend (RM band → USD midpoint)
df['avg_spend_per_visit_usd'] = df['avg_spend_per_visit'].map(
    {'Less than RM20':10,'Around RM20 - RM40':30,'More than RM40':50,'Zero':0}) * 0.21

# Visit-duration midpoints (minutes)
df['visit_duration_minutes'] = df['visit_duration'].map({
    'Below 30 minutes':15,'Between 30 minutes to 1 hour':45,
    'Between 1 hour to 2 hours':90,'Between 2 hours to 3 hours':150,
    'More than 3 hours':210})

# Ordered visit frequency
visit_order = [c for c in ['Never','Rarely','Monthly','Weekly','Daily']
               if c in df['visit_frequency'].unique()]
df['visit_frequency'] = pd.Categorical(df['visit_frequency'],
                                       categories=visit_order, ordered=True)

print(f"Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## 3 · How variables relate to each other

**What it shows.** Pearson correlations across the eight numeric survey signals.

**Why it matters.** It tells us which experience dimensions move together and which are independent — useful for feature selection and storytelling.

**Key insights**
- **Service, Ambiance and Quality form a tight cluster** (r ≈ 0.45 – 0.60): customers who love one love the others.
- **Spend is most correlated with visit frequency** (r ≈ 0.49) and **age** (r ≈ 0.35) — older, more frequent visitors spend more.
- **Promo importance is almost independent of spend** — discounts don't drive bigger baskets.

![01_correlation_heatmap](figures/01_correlation_heatmap.png)

In [ ]:
df['_age_code']   = df['age_group'].cat.codes
df['_visit_code'] = df['visit_frequency'].cat.codes
corr_cols  = ['avg_spend_per_visit_usd','_visit_code','quality_rating',
              'price_rating','promo_importance','ambiance_rating',
              'wifi_quality','service_rating','_age_code']
labels     = ['Spend (USD)','Visit freq.','Quality','Price','Promo importance',
              'Ambiance','WiFi','Service','Age group']
corr = df[corr_cols].corr(); corr.index = labels; corr.columns = labels

fig, ax = plt.subplots(figsize=(9, 7.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=DIV_CMAP,
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            cbar_kws={"shrink":0.8, "label":"Pearson r"}, ax=ax)
ax.set_title("How Survey Variables Relate to One Another", pad=12)
plt.show()

## 4 · Customer demographics

**What it shows.** The mix of age groups in the sample (donut chart with absolute counts and shares).

**Why it matters.** Survey results are only as representative as the sample. Knowing the dominant age cohort frames every downstream insight.

**Key insight.** **~70 % of respondents are Young Adults (20-29)** — Starbucks Malaysia's voice-of-customer is overwhelmingly Gen Z / early Millennials.

![02_age_distribution](figures/02_age_distribution.png)

In [ ]:
age_counts = df['age_group'].value_counts().reindex(age_order)
fig, ax = plt.subplots(figsize=(8, 6))
wedges, _, autotexts = ax.pie(
    age_counts, labels=age_counts.index, startangle=90,
    colors=PALETTE[:len(age_counts)],
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2),
    autopct=lambda p: f"{p:.0f}%\n({int(round(p*age_counts.sum()/100))})",
    pctdistance=0.78)
for t in autotexts: t.set_color("white"); t.set_fontweight("bold")
ax.set_title("Customer Age Group Distribution", pad=10)
ax.text(0,0,f"n = {age_counts.sum()}", ha="center", va="center",
        fontsize=14, fontweight="bold", color=SB_DARK_GREEN)
plt.show()

## 5 · Average ratings across age groups

**What it shows.** Mean rating (1-5) given by each age group across the six experience dimensions.

**Why it matters.** Identifies which segments are most / least satisfied with each lever.

**Key insights**
- **Adults (30-39) are the happiest cohort** across most dimensions, especially Service.
- **WiFi underperforms in every age group** (avg ≈ 2.7-3.5) — the weakest experience lever overall.
- **Teens rate Price the lowest** — they're the most price-sensitive segment.

![03_ratings_by_age](figures/03_ratings_by_age.png)

In [ ]:
rating_cols = ['quality_rating','price_rating','promo_importance',
               'ambiance_rating','wifi_quality','service_rating']
nice = ['Quality','Price','Promo','Ambiance','WiFi','Service']
avg_age = df.groupby('age_group', observed=True)[rating_cols].mean()
avg_age.columns = nice

fig, ax = plt.subplots(figsize=(11, 6))
avg_age.plot(kind="bar", ax=ax, color=PALETTE[:6], width=0.82, edgecolor="white")
ax.set_title("Average Customer Ratings by Age Group", pad=10)
ax.set_xlabel(""); ax.set_ylabel("Average rating (1–5)"); ax.set_ylim(0, 5)
ax.legend(title="Dimension", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0); plt.show()

## 6 · Spend per visit by age

**What it shows.** Distribution of average spend per visit (USD) by age group — boxplot for spread, strip for individual responses.

**Why it matters.** Pinpoints the highest-value cohorts.

**Key insights**
- **Mature Adults (40+) and Adults (30-39) have the highest median spend.**
- **Teens are concentrated at the low end** — typically the *<RM20* band.

![04_spend_by_age](figures/04_spend_by_age.png)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.boxplot(x='age_group', y='avg_spend_per_visit_usd', data=df,
            order=age_order, palette=PALETTE[:4], width=0.55,
            fliersize=0, ax=ax)
sns.stripplot(x='age_group', y='avg_spend_per_visit_usd', data=df,
              order=age_order, color=SB_DARK_GREEN, alpha=0.35,
              size=3.5, jitter=0.25, ax=ax)
ax.set_title("Spend per Visit by Age Group (USD)", pad=10)
ax.set_xlabel(""); ax.set_ylabel("Avg spend per visit (USD)")
plt.show()

## 7 · Visit frequency mix by age

**What it shows.** Stacked-percentage view of how often each age group visits.

**Why it matters.** Frequency is a leading indicator of LTV.

**Key insight.** **Adults (30-39) are the only group with a meaningful share of weekly+ visitors** — the most engaged cohort.

![05_visit_freq_by_age](figures/05_visit_freq_by_age.png)

In [ ]:
visit_age = pd.crosstab(df['age_group'], df['visit_frequency'])
visit_age_pct = visit_age.div(visit_age.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 6))
visit_age_pct.plot(kind="bar", stacked=True, ax=ax,
                   color=PALETTE[:visit_age_pct.shape[1]],
                   width=0.7, edgecolor="white")
ax.set_title("Visit Frequency Mix by Age Group", pad=10)
ax.set_ylabel("Share of customers (%)"); ax.set_xlabel("")
ax.legend(title="Visit frequency", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0); plt.show()

## 8 · Loyalty (continue buying) by age

**What it shows.** Share of each age group who say they will keep buying at Starbucks.

**Why it matters.** Single most important business outcome in the survey.

**Key insight.** **Loyalty is high (>75 %) in every age group**, but **Mature Adults are the most loyal** segment.

![06_loyalty_by_age](figures/06_loyalty_by_age.png)

In [ ]:
loyalty_age = pd.crosstab(df['age_group'], df['continue_buying'])
loyalty_age_pct = loyalty_age.div(loyalty_age.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5.5))
loyalty_age_pct[['Yes','No']].plot(kind="barh", stacked=True, ax=ax,
    color=[SB_GREEN, SB_RED], width=0.65, edgecolor="white")
for i, (yes, no) in enumerate(loyalty_age_pct[['Yes','No']].values):
    ax.text(yes/2, i, f"{yes:.0f}%", ha="center", va="center",
            color="white", fontweight="bold", fontsize=11)
ax.set_xlim(0, 100)
ax.set_title("Loyalty (Continue Buying) by Age Group", pad=10)
ax.set_xlabel("Share of customers (%)"); ax.set_ylabel("")
ax.legend(title="Continue buying?", loc="lower right")
plt.show()

## 9 · Employment status

**What it shows.** Occupation mix of respondents.

**Why it matters.** Employment status is a strong proxy for disposable income and visit context (study vs. work breaks).

**Key insight.** The sample is dominated by **students and employed professionals**.

![07_status_distribution](figures/07_status_distribution.png)

In [ ]:
status_counts = df['current_status'].value_counts()
fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.barh(status_counts.index[::-1], status_counts.values[::-1],
               color=PALETTE[:len(status_counts)][::-1], edgecolor="white")
for bar, v in zip(bars, status_counts.values[::-1]):
    pct = v / status_counts.sum() * 100
    ax.text(v + max(status_counts)*0.01, bar.get_y()+bar.get_height()/2,
            f"{v} ({pct:.0f}%)", va="center", fontsize=10, color="#333")
ax.set_title("Customer Current Status", pad=10)
ax.set_xlabel("Number of respondents"); ax.set_ylabel("")
plt.show()

## 10 · How long customers stay (frequency × visit type)

**What it shows.** Average minutes spent in store, broken down by visit frequency and visit type (dine-in, take-away, drive-through).

**Why it matters.** Store-occupancy planning and seat-turnover decisions.

**Key insight.** **Dine-in customers stay 2-3× longer than take-away** customers across every frequency band — capacity planning should weigh dine-in heavily.

![10_duration_heatmap](figures/10_duration_heatmap.png)

In [ ]:
dur = df.groupby(['visit_frequency','visit_type'], observed=True
                 )['visit_duration_minutes'].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(dur, annot=True, fmt=".0f", cmap=SEQ_CMAP,
            linewidths=0.6, linecolor="white",
            cbar_kws={"label":"Avg duration (minutes)"}, ax=ax)
ax.set_title("Average Visit Duration by Frequency × Visit Type", pad=10)
ax.set_xlabel("Visit type"); ax.set_ylabel("Visit frequency")
plt.show()

## 11 · Distribution of ratings (small multiples)

**What it shows.** Histogram of every 1-5 rating, side-by-side, on a shared y-axis.

**Why it matters.** Means hide skew; this view exposes which dimensions have unhappy tails.

**Key insights**
- **Service & Ambiance** are positively skewed → mostly 4-5 ratings (best-performing levers).
- **WiFi & Price** are centered on 3 with meaningful 1-2 tails (worst-performing levers).

![13_rating_distributions](figures/13_rating_distributions.png)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
for ax, col, label in zip(axes.flat, rating_cols, nice):
    sns.countplot(x=col, data=df, color=SB_GREEN, ax=ax, edgecolor="white")
    ax.set_title(label, color=SB_DARK_GREEN)
    ax.set_xlabel("Rating (1–5)"); ax.set_ylabel("Count")
    annotate_bars(ax, offset=2)
fig.suptitle("Distribution of Customer Ratings Across Six Dimensions",
             fontsize=15, fontweight="bold", color=SB_DARK_GREEN, y=1.01)
fig.tight_layout(); plt.show()

## 12 · The headline number — will they continue buying?

**What it shows.** Overall share of customers planning to keep buying.

**Why it matters.** It's the closest thing to a Net-Loyalty signal in the survey.

**Key insight.** **~80 % of respondents plan to continue buying at Starbucks** — strong baseline loyalty.

![16_loyalty_donut](figures/16_loyalty_donut.png)

In [ ]:
loyalty = df['continue_buying'].value_counts()
fig, ax = plt.subplots(figsize=(7, 6))
wedges, _, autotexts = ax.pie(
    loyalty, labels=loyalty.index,
    colors=[SB_GREEN if k=="Yes" else SB_RED for k in loyalty.index],
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2),
    autopct=lambda p: f"{p:.1f}%", pctdistance=0.78)
for t in autotexts: t.set_color("white"); t.set_fontweight("bold")
ax.set_title("Will Customers Continue Buying at Starbucks?", pad=10)
ax.text(0,0,f"n = {loyalty.sum()}", ha="center", va="center",
        fontsize=13, fontweight="bold", color=SB_DARK_GREEN)
plt.show()

## 13 · What separates loyal from non-loyal customers?

**What it shows.** Mean rating per dimension, side-by-side, for the *Yes* and *No* groups.

**Why it matters.** Highlights the levers with the largest loyalty-gap.

**Key insights**
- **Quality, Service and Ambiance show the biggest gap** — loyal customers rate these noticeably higher.
- **Price ratings barely differ** → price isn't the main loyalty lever.

![18_ratings_loyal_vs_not](figures/18_ratings_loyal_vs_not.png)

In [ ]:
mean_by_loy = df.groupby('continue_buying')[rating_cols].mean().T
mean_by_loy.index = nice

fig, ax = plt.subplots(figsize=(10, 5.5))
mean_by_loy.plot(kind="bar", ax=ax, color=[SB_RED, SB_GREEN],
                 width=0.7, edgecolor="white")
ax.set_title("Average Ratings: Loyal vs Non-Loyal Customers", pad=10)
ax.set_xlabel(""); ax.set_ylabel("Average rating (1–5)"); ax.set_ylim(0, 5)
plt.xticks(rotation=0); ax.legend(title="Continue buying?")
annotate_bars(ax, fmt="{:.2f}"); plt.show()

## 14 · Predicting loyalty with ML

**What it shows.** Confusion matrices for three classifiers predicting `continue_buying`.

**Why it matters.** Validates whether survey signals are *predictive*, not just *descriptive*.

**Key insight.** Logistic Regression generalises best on this small sample (n = 122) — tree models slightly **over-fit** because of the class imbalance and limited rows.

![21_confusion_matrices](figures/21_confusion_matrices.png)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

m_df = df.drop(columns=['timestamp','_age_code','_visit_code',
                        'avg_spend_per_visit_usd','visit_duration_minutes']).copy()
le_target = LabelEncoder()
y = le_target.fit_transform(m_df['continue_buying'])
X = m_df.drop(columns=['continue_buying'])

for col in X.select_dtypes(include=['object','category']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
num_cols = X.select_dtypes(include=['int64','float64','int32']).columns
X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
X[num_cols] = StandardScaler().fit_transform(X[num_cols])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y,
                                          random_state=42, test_size=0.2)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42,
                             use_label_encoder=False),
}
results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr); pred = model.predict(X_te)
    results[name] = dict(acc=accuracy_score(y_te, pred),
                         cm=confusion_matrix(y_te, pred))
    print(f"\n{name}: acc = {results[name]['acc']:.2%}")
    print(classification_report(y_te, pred, target_names=le_target.classes_))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(r['cm'], annot=True, fmt='d', cmap=SEQ_CMAP,
                xticklabels=le_target.classes_, yticklabels=le_target.classes_,
                cbar=False, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_title(f"{name}\nacc = {r['acc']:.2%}", color=SB_DARK_GREEN)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.suptitle("Confusion Matrices — Predicting Customer Loyalty",
             fontsize=15, fontweight="bold", color=SB_DARK_GREEN, y=1.04)
fig.tight_layout(); plt.show()

## 15 · Model accuracy comparison

**What it shows.** Test-set accuracy of the three classifiers on the same hold-out split.

**Why it matters.** Picks the production-ready model.

**Key insight.** **Logistic Regression wins (88 %)** — a simple, interpretable model is enough for this dataset.

![22_model_accuracy](figures/22_model_accuracy.png)

In [ ]:
acc_df = pd.Series({k: v['acc'] for k, v in results.items()}).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(acc_df.index, acc_df.values,
               color=[SB_GOLD, SB_DARK_GREEN, SB_GREEN], edgecolor="white")
for bar, v in zip(bars, acc_df.values):
    ax.text(v+0.005, bar.get_y()+bar.get_height()/2, f"{v:.2%}",
            va="center", fontsize=11, fontweight="bold", color=SB_DARK_GREEN)
ax.set_xlim(0, 1.05)
ax.set_title("Model Accuracy Comparison — Loyalty Prediction", pad=10)
ax.set_xlabel("Test accuracy"); ax.set_ylabel("")
plt.show()

## 16 · Suggested next steps & missing visualisations

| Suggestion | Why it would help |
|---|---|
| **Driver analysis (logistic-regression coefficients)** | Quantify which rating moves loyalty the most — a natural follow-on to the model. |
| **Radar / spider chart per segment** | Compare the 6-dimension experience profile of *Loyal vs Non-Loyal* in one glance. |
| **Promo-channel heatmap** (channel × age) | Inform marketing-spend allocation; the raw column already exists (`promo_channels`). |
| **NPS-style stacked bar** for ratings | Recode 1-2 = Detractors, 3 = Passives, 4-5 = Promoters per dimension. |
| **Bootstrap CI on the loyalty %** | Show the headline number with uncertainty (n is only 122). |

## 17 · Summary

- Sample is **small (n = 122)** and **skewed to Young Adults** — generalise with care.
- **Service, Ambiance and Quality drive loyalty**; **WiFi and Price are the weakest experience levers**.
- **Older + more frequent customers spend more** — natural high-LTV segment.
- **80 % loyalty** is high; a **simple Logistic Regression predicts it at 88 % accuracy**.
